# Hand-picked 50 LEMPRO / LangPro rerun

This notebook reruns the manually curated 50-problem experiment from Appendix C of the paper draft: 25 SICK examples and 25 SNLI examples. It uses the repository's `kbprojection` pipeline rather than a separate script:

1. Load the exact hand-picked problems.
2. Run LangPro without injected KB.
3. Ask the configured LLM for candidate lexical relations.
4. Normalize/filter those relations into LangPro-compatible KB injections.
5. Run LangPro again with raw and filtered KB.
6. Save JSONL/CSV outputs for analysis and manual validation.

The notebook defaults to `RUN_EXPERIMENT = False` so opening it does not make live LangPro or LLM calls by accident.

## 1. Imports

In [1]:
import csv
import importlib.util
import json
import os
import re
import sys
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from pprint import pprint

if sys.version_info < (3, 10):
    raise RuntimeError("Use a Python 3.10+ kernel for this notebook.")

# Runtime/path setup. In Colab, set KBPROJECTION_RUNTIME=colab and
# KBPROJECTION_PROJECT_ROOT=/content/drive/MyDrive/kbprojection before this cell,
# or let configure_runtime auto-detect Colab and mount Drive.
def _find_package_root(start: Path) -> Path | None:
    start = start.expanduser().resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "kbprojection" / "__init__.py").exists():
            return candidate
        nested = candidate / "kbprojection"
        if (nested / "kbprojection" / "__init__.py").exists():
            return nested
    return None

_candidate_roots = [
    Path(os.environ.get("KBPROJECTION_PROJECT_ROOT", Path.cwd())).expanduser().resolve(),
    Path.cwd().resolve(),
    Path("/content/drive/MyDrive/kbprojection"),
]
_package_roots = []
for _candidate_root in _candidate_roots:
    _package_root = _find_package_root(_candidate_root)
    if _package_root and _package_root not in _package_roots:
        _package_roots.append(_package_root)
for _package_root in reversed(_package_roots):
    if str(_package_root) in sys.path:
        sys.path.remove(str(_package_root))
    sys.path.insert(0, str(_package_root))

# If Python cached the outer repository folder as a namespace package, clear it.
_cached_kbp = sys.modules.get("kbprojection")
if _cached_kbp is not None and not getattr(_cached_kbp, "__file__", None):
    del sys.modules["kbprojection"]

import kbprojection
from kbprojection import SICKLoader, SNLILoader
from kbprojection.runtime import configure_runtime
from kbprojection.models import ExperimentResult, ProblemConfig, TestMode
from kbprojection.orchestration import experiment_result_to_json, process_single_problem

RUNTIME_PATHS = configure_runtime()
PROJECT_ROOT = RUNTIME_PATHS.project_root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("kbprojection version:", kbprojection.__version__)
print("Runtime:", RUNTIME_PATHS.runtime)
print("Project root:", PROJECT_ROOT)


kbprojection version: 0.4.1
Runtime: local
Project root: /Users/jorrytdejong/Documents/RNL paper publication


## 2. Experiment settings

Edit this cell before a real run. Recommended workflow:

- First set `RUN_EXPERIMENT = False` and run through the validation cells.
- Then set `RUN_EXPERIMENT = True` with `DRY_RUN_LIMIT = 2`.
- Finally set `DRY_RUN_LIMIT = None` for the full 50 examples.

Use `RUN_ABLATION = True` only after the main run works, because it can multiply LangPro calls.

In [2]:
# Safety switch: set True only when ready for live LangPro + LLM calls.
RUN_EXPERIMENT = True

# Use a small number such as 2 for a smoke test, or None for all 50.
DRY_RUN_LIMIT = 2

# LLM settings. Provider may also be auto-detected if you set LLM_PROVIDER = None.
LLM_PROVIDER = "openai"       # "openai", "openrouter", "gemini", "claude", or None
MODEL = "gpt-5.4"
PROMPT_STYLE = "icl"

# Pipeline settings.
TEST_MODE = "both"            # "both" tests raw and filtered KB; good for reproduction tables.
RUN_ABLATION = False
POST_PROCESS = True
VERBOSE = True

# LangPro endpoint used by kbprojection.langpro.langpro_api_call.
LANGPRO_ENDPOINT = "https://langpro-annotator.hum.uu.nl/langpro-api/prove/"

# Local data/output locations.
# This fallback lets the settings cell run even if the imports/runtime cell was not run first.
if "RUNTIME_PATHS" not in globals():
    from kbprojection.runtime import configure_runtime
    RUNTIME_PATHS = configure_runtime()

DATA_DIR = RUNTIME_PATHS.data_dir
RESULTS_DIR = RUNTIME_PATHS.results_dir / "handpicked_50"
CACHE_ROOT = RUNTIME_PATHS.cache_root / "handpicked_50"

# Give each model/config its own cache namespace to avoid accidental cross-model reuse.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
MODEL_SLUG = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(MODEL))
RUN_NAME = f"{RUN_ID}_{MODEL_SLUG}_{PROMPT_STYLE}_{TEST_MODE}"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = CACHE_ROOT / RUN_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Run name:", RUN_NAME)
print("Results dir:", RESULTS_DIR)
print("Cache dir:", CACHE_DIR)

Run name: 20260430_160702_gpt-5.4_icl_both
Results dir: /Users/jorrytdejong/Documents/RNL paper publication/experiment_results/handpicked_50
Cache dir: /Users/jorrytdejong/Documents/RNL paper publication/experiment_cache/handpicked_50/20260430_160702_gpt-5.4_icl_both


## 3. Hand-picked problem IDs

These are transcribed from Appendix C.1 of the PDF draft. Keeping the split next to each ID matters, especially for SICK.

In [3]:
HANDPICKED = [
    # SICK train, 15 examples
    {"dataset": "sick", "split": "train", "id": "1792"},
    {"dataset": "sick", "split": "train", "id": "2281"},
    {"dataset": "sick", "split": "train", "id": "2809"},
    {"dataset": "sick", "split": "train", "id": "3853"},
    {"dataset": "sick", "split": "train", "id": "4181"},
    {"dataset": "sick", "split": "train", "id": "4766"},
    {"dataset": "sick", "split": "train", "id": "4974"},
    {"dataset": "sick", "split": "train", "id": "5149"},
    {"dataset": "sick", "split": "train", "id": "5198"},
    {"dataset": "sick", "split": "train", "id": "5435"},
    {"dataset": "sick", "split": "train", "id": "5462"},
    {"dataset": "sick", "split": "train", "id": "5554"},
    {"dataset": "sick", "split": "train", "id": "5555"},
    {"dataset": "sick", "split": "train", "id": "6584"},
    {"dataset": "sick", "split": "train", "id": "8624"},
    # SICK dev, 1 example
    {"dataset": "sick", "split": "dev", "id": "3586"},
    # SICK test, 9 examples
    {"dataset": "sick", "split": "test", "id": "1262"},
    {"dataset": "sick", "split": "test", "id": "1497"},
    {"dataset": "sick", "split": "test", "id": "3641"},
    {"dataset": "sick", "split": "test", "id": "3974"},
    {"dataset": "sick", "split": "test", "id": "4297"},
    {"dataset": "sick", "split": "test", "id": "4494"},
    {"dataset": "sick", "split": "test", "id": "4589"},
    {"dataset": "sick", "split": "test", "id": "4972"},
    {"dataset": "sick", "split": "test", "id": "5230"},
    # SNLI train, 25 examples
    {"dataset": "snli", "split": "train", "id": "3629664676.jpg#4r1e"},
    {"dataset": "snli", "split": "train", "id": "vg_len26r4e"},
    {"dataset": "snli", "split": "train", "id": "vg_len66r1e"},
    {"dataset": "snli", "split": "train", "id": "3159569570.jpg#4r1e"},
    {"dataset": "snli", "split": "train", "id": "4294390957.jpg#3r1e"},
    {"dataset": "snli", "split": "train", "id": "1184967930.jpg#4r4e"},
    {"dataset": "snli", "split": "train", "id": "2521878609.jpg#4r1e"},
    {"dataset": "snli", "split": "train", "id": "303607405.jpg#4r3e"},
    {"dataset": "snli", "split": "train", "id": "436393371.jpg#3r1e"},
    {"dataset": "snli", "split": "train", "id": "vg_len84r1e"},
    {"dataset": "snli", "split": "train", "id": "326456451.jpg#4r1e"},
    {"dataset": "snli", "split": "train", "id": "vg_len120r5e"},
    {"dataset": "snli", "split": "train", "id": "2543017787.jpg#4r1e"},
    {"dataset": "snli", "split": "train", "id": "2647049174.jpg#4r1e"},
    {"dataset": "snli", "split": "train", "id": "2618866067.jpg#4r1e"},
    {"dataset": "snli", "split": "train", "id": "vg_len47r3e"},
    {"dataset": "snli", "split": "train", "id": "vg_len47r4e"},
    {"dataset": "snli", "split": "train", "id": "vg_len46r5e"},
    {"dataset": "snli", "split": "train", "id": "3228793611.jpg#4r1e"},
    {"dataset": "snli", "split": "train", "id": "3134092148.jpg#3r2e"},
    {"dataset": "snli", "split": "train", "id": "2217728745.jpg#4r1e"},
    {"dataset": "snli", "split": "train", "id": "3479245321.jpg#4r1e"},
    {"dataset": "snli", "split": "train", "id": "2741990005.jpg#4r1e"},
    {"dataset": "snli", "split": "train", "id": "3614595423.jpg#4r1e"},
    {"dataset": "snli", "split": "train", "id": "207584893.jpg#3r1e"},
]

assert len(HANDPICKED) == 50
print(Counter(item["dataset"] for item in HANDPICKED))
print(Counter((item["dataset"], item["split"]) for item in HANDPICKED))

Counter({'sick': 25, 'snli': 25})
Counter({('snli', 'train'): 25, ('sick', 'train'): 15, ('sick', 'test'): 9, ('sick', 'dev'): 1})


## 4. Preflight checks

This cell checks packages, API keys, and loads the requested dataset splits. Loading SNLI train can take a little while.

In [4]:
def has_package(name: str) -> bool:
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:
        return False

provider_env = {
    "openai": "OPENAI_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "claude": "ANTHROPIC_API_KEY",
    None: None,
}

print("Package checks:")
for package in ["requests", "pydantic", "nltk", "openai", "anthropic", "google.genai"]:
    print(f"  {package:14s}", "OK" if has_package(package) else "missing")

key_name = provider_env.get(LLM_PROVIDER)
print("\nProvider:", LLM_PROVIDER)
if key_name:
    print(f"API key {key_name}:", "set" if os.environ.get(key_name) else "not set")
elif LLM_PROVIDER is None:
    detected = [name for name in provider_env.values() if name and os.environ.get(name)]
    print("Auto-detectable provider keys:", detected or "none")
else:
    print("Unknown provider; expected one of:", sorted(k for k in provider_env if k))

sick_loader = SICKLoader(data_dir=DATA_DIR / "sick")
snli_loader = SNLILoader(data_dir=DATA_DIR / "snli")
loaders = {"sick": sick_loader, "snli": snli_loader}

needed_splits = defaultdict(set)
for item in HANDPICKED:
    needed_splits[item["dataset"]].add(item["split"])

for dataset_name, splits in needed_splits.items():
    print(f"\nLoading {dataset_name}: {sorted(splits)}")
    loaders[dataset_name].load(splits=sorted(splits))

Package checks:
  requests       OK
  pydantic       OK
  nltk           OK
  openai         OK
  anthropic      OK
  google.genai   OK

Provider: openai
API key OPENAI_API_KEY: set

Loading sick: ['dev', 'test', 'train']
[SICKLoader] Loaded 500 problems from dev.
[SICKLoader] Loaded 4927 problems from test.
[SICKLoader] Loaded 4500 problems from train.

Loading snli: ['train']
[SNLILoader] Loaded 549367 problems from train.


## 5. Validate the fixed problem set

All 50 should be present and entailment-labeled. If an ID fails here, fix the ID before running the expensive cells.

In [5]:
def get_problem(item):
    return loaders[item["dataset"]].get_problem(item["id"], split=item["split"])

problems = []
missing = []
for item in HANDPICKED:
    try:
        prob = get_problem(item)
        problems.append(prob)
    except Exception as exc:
        missing.append((item, repr(exc)))

print("Loaded problems:", len(problems))
print("Missing:", len(missing))
if missing:
    pprint(missing)

labels = Counter(prob.gold_label.value for prob in problems)
print("Labels:", labels)
assert not missing
assert len(problems) == 50
assert set(labels) == {"entailment"}, "The paper's 50-problem reproduction set should be entailment-only."

for prob in problems[:5]:
    print("\n", prob.dataset, prob.split, prob.id)
    print("P:", " ".join(prob.premises))
    print("H:", prob.hypothesis)

Loaded problems: 50
Missing: 0
Labels: Counter({'entailment': 50})

 sick train 1792
P: A lion is slowly walking
H: A lion is slowly moving around

 sick train 2281
P: A dog is opening a can of food
H: A dog is biting a can

 sick train 2809
P: A cartoon airplane is landing
H: An animated airplane is landing

 sick train 3853
P: A cat is playing a piano
H: A cat is playing keyboards

 sick train 4181
P: A dog is eating a doll
H: A dog is biting a doll


## 6. Patch LangPro endpoint

`process_single_problem` uses a module-level `langpro_api_call`. This patch makes the notebook's `LANGPRO_ENDPOINT` setting explicit.

In [6]:
import kbprojection.orchestration as orchestration
from kbprojection.langpro import langpro_api_call as original_langpro_api_call

def langpro_api_call_with_notebook_endpoint(premises, hypothesis, **kwargs):
    kwargs.setdefault("endpoint", LANGPRO_ENDPOINT)
    return original_langpro_api_call(premises, hypothesis, **kwargs)

orchestration.langpro_api_call = langpro_api_call_with_notebook_endpoint
print("LangPro endpoint:", LANGPRO_ENDPOINT)

LangPro endpoint: https://langpro-annotator.hum.uu.nl/langpro-api/prove/


## 7. Run the 50-problem experiment

This cell is the expensive one. It writes one cache file per problem and one combined JSONL/CSV after the loop.

In [9]:
if "TestMode" not in globals():
    from kbprojection.models import TestMode

config = ProblemConfig(
    llm_provider=LLM_PROVIDER,
    model=MODEL,
    prompt_style=PROMPT_STYLE,
    post_process=POST_PROCESS,
    test_mode=TestMode(TEST_MODE),
    run_ablation=RUN_ABLATION,
    verbose=VERBOSE,
)

print("Config:")
pprint(config.model_dump())

run_items = HANDPICKED[:DRY_RUN_LIMIT] if DRY_RUN_LIMIT is not None else HANDPICKED
print(f"Will process {len(run_items)} / {len(HANDPICKED)} problems")

if not RUN_EXPERIMENT:
    raise RuntimeError("Set RUN_EXPERIMENT = True when you are ready to call LangPro and the LLM.")

results = []
for idx, item in enumerate(run_items, start=1):
    prob = get_problem(item)
    safe_id = prob.id.replace("/", "_").replace("#", "_")
    cache_file = CACHE_DIR / f"{idx:02d}_{prob.dataset}_{prob.split}_{safe_id}.json"

    if cache_file.exists():
        try:
            result = ExperimentResult.model_validate_json(cache_file.read_text(encoding="utf-8"))
            print(f"\n[{idx:02d}/{len(run_items)}] {prob.dataset}/{prob.split}/{prob.id} [cached] -> {result.final_status}")
        except Exception as exc:
            print(f"\n[{idx:02d}/{len(run_items)}] {prob.dataset}/{prob.split}/{prob.id} [cache unreadable: {exc}; reprocessing]")
            result = await process_single_problem(prob, config=config, cache_file=cache_file)
    else:
        print(f"\n[{idx:02d}/{len(run_items)}] {prob.dataset}/{prob.split}/{prob.id}")
        result = await process_single_problem(prob, config=config, cache_file=cache_file)

    results.append(result)

print("\nDone. Results:", len(results))

Config:
{'llm_provider': 'openai',
 'model': 'gpt-5.4',
 'post_process': True,
 'prompt_style': 'icl',
 'run_ablation': False,
 'test_mode': <TestMode.BOTH: 'both'>,
 'verbose': True}
Will process 2 / 50 problems

[01/2] sick/train/1792 [cached] -> ExperimentStatus.RAW_KB_SOLVED

[02/2] sick/train/2281 [cached] -> ExperimentStatus.KB_GENERATION_EMPTY

Done. Results: 2


## 8. Save flattened JSONL and CSV

In [10]:
def value(x):
    return x.value if hasattr(x, "value") else x

def flatten_result(result: ExperimentResult) -> dict:
    prob = result.problem
    return {
        "dataset": prob.dataset,
        "split": prob.split,
        "id": prob.id,
        "gold_label": value(prob.gold_label),
        "premise": " ".join(prob.premises),
        "hypothesis": prob.hypothesis,
        "pred_no_kb": value(result.pred_no_kb),
        "pred_with_raw_kb": value(result.pred_with_raw_kb),
        "pred_with_kb": value(result.pred_with_kb),
        "status_no_kb": value(result.status_no_kb),
        "status_with_raw_kb": value(result.status_with_raw_kb),
        "status_with_kb": value(result.status_with_kb),
        "final_status": value(result.final_status),
        "fixed_by": result.fixed_by,
        "kb_raw": json.dumps(result.kb_raw or [], ensure_ascii=False),
        "kb_filtered": json.dumps(result.kb_filtered or [], ensure_ascii=False),
        "essential_kb": json.dumps(result.essential_kb or [], ensure_ascii=False),
        "kb_details": json.dumps([d.model_dump() for d in (result.kb_details or [])], ensure_ascii=False),
    }

rows = [flatten_result(r) for r in results]

jsonl_path = RESULTS_DIR / f"{RUN_NAME}.jsonl"
csv_path = RESULTS_DIR / f"{RUN_NAME}.csv"

with jsonl_path.open("w", encoding="utf-8") as f:
    for result in results:
        f.write(experiment_result_to_json(result) + "\n")

with csv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else [])
    if rows:
        writer.writeheader()
        writer.writerows(rows)

print("JSONL:", jsonl_path)
print("CSV:", csv_path)

JSONL: /Users/jorrytdejong/Documents/RNL paper publication/experiment_results/handpicked_50/20260428_201319_gpt-5-mini_icl_both.jsonl
CSV: /Users/jorrytdejong/Documents/RNL paper publication/experiment_results/handpicked_50/20260428_201319_gpt-5-mini_icl_both.csv


## 9. Summary tables

In [11]:
def count_by(key):
    return Counter(row[key] for row in rows)

print("Overall final statuses:")
pprint(count_by("final_status"))

print("\nFixed by:")
pprint(count_by("fixed_by"))

print("\nBy dataset and final status:")
by_dataset_status = Counter((row["dataset"], row["final_status"]) for row in rows)
pprint(by_dataset_status)

print("\nNo-KB baseline labels:")
pprint(count_by("pred_no_kb"))

fixed_rows = [row for row in rows if str(row["final_status"]).startswith("fixed")]
print(f"\nFixed: {len(fixed_rows)} / {len(rows)}")

Overall final statuses:
Counter({'error_no_kb': 2})

Fixed by:
Counter({None: 2})

By dataset and final status:
Counter({('sick', 'error_no_kb'): 2})

No-KB baseline labels:
Counter({'-': 2})

Fixed: 0 / 2


## 10. Manual validation sheet

LangPro proof closure is not the same as semantic validity. This cell creates a review CSV with empty columns for human checking.

In [12]:
review_path = RESULTS_DIR / f"{RUN_NAME}_manual_validation.csv"
review_fields = [
    "dataset", "split", "id", "premise", "hypothesis", "gold_label",
    "pred_no_kb", "pred_with_kb", "final_status", "fixed_by",
    "kb_filtered", "essential_kb",
    "semantic_valid", "uses_premise", "notes",
]

with review_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=review_fields)
    writer.writeheader()
    for row in rows:
        writer.writerow({field: row.get(field, "") for field in review_fields})

print("Manual validation CSV:", review_path)

Manual validation CSV: /Users/jorrytdejong/Documents/RNL paper publication/experiment_results/handpicked_50/20260428_201319_gpt-5-mini_icl_both_manual_validation.csv


## 11. Inspect individual cases

In [13]:
def show_case(result: ExperimentResult):
    prob = result.problem
    print(f"{prob.dataset}/{prob.split}/{prob.id}")
    print("Gold:", value(prob.gold_label))
    print("P:", " ".join(prob.premises))
    print("H:", prob.hypothesis)
    print("No KB:", value(result.pred_no_kb), "| Raw KB:", value(result.pred_with_raw_kb), "| Filtered KB:", value(result.pred_with_kb))
    print("Final:", value(result.final_status), "fixed_by:", result.fixed_by)
    print("Raw KB:", result.kb_raw)
    print("Filtered KB:", result.kb_filtered)
    if result.essential_kb:
        print("Essential KB:", result.essential_kb)

# Change the index to inspect another case.
if results:
    show_case(results[0])

sick/train/1792
Gold: entailment
P: A lion is slowly walking
H: A lion is slowly moving around
No KB: - | Raw KB: None | Filtered KB: None
Final: error_no_kb fixed_by: None
Raw KB: None
Filtered KB: None


## Recommended interpretation

For the paper rerun, report at least these quantities:

- How many of the 50 were already solved by LangPro without KB.
- How many previously unsolved cases were fixed by raw KB, filtered KB, or both.
- How many filtered KBs are semantically valid after manual inspection.
- Per-dataset counts for SICK and SNLI.
- If comparing models, rerun the notebook once per model and compare the generated CSVs, using the same prompt style and post-processing setting.

Keep the cache directory with the results, because it is the audit trail for individual proof attempts and generated KB injections.